In [1]:
import pandas as pd
import numpy as np
import re
import datetime
import requests
import itertools
import os

# NLP text processing - not very important
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter
import spacy

# random forest, LightGBM, XGBoost
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Logistic regression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImPipeline
from sklearn.inspection import permutation_importance

# LR enhancement
from category_encoders import WOEEncoder
from sklearn.preprocessing import PolynomialFeatures

# Feature selection and Param tuning
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from sklearn.svm import LinearSVC
from sklearn.base import clone
from pandas.api.types import CategoricalDtype
from sklearn.model_selection import GridSearchCV

# caching
import joblib

# data visualization
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

In [2]:
# Load the English language model
nlp = spacy.load("en_core_web_sm")

In [3]:
# util print function
def print_full(x):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    pd.set_option('display.float_format', '{:20,.2f}'.format)
    pd.set_option('display.max_colwidth', None)
    print(x)
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.float_format')
    pd.reset_option('display.max_colwidth')

In [4]:
# Load Excel file
excel_file = pd.ExcelFile('data/Steiling_H44003_datav2.xlsx')
excel_file2 = pd.ExcelFile('data/Steiling_H44003_ct.xlsx')

In [5]:
demographic = pd.read_excel(excel_file, 'SteilingH44003_dem')
comorb_problist = pd.read_excel(excel_file, 'SteilingH44003_comorbproblist')
comorb_encounter = pd.read_excel(excel_file,  'SteilingH44003_comorbencounter')
tobaco_use = pd.read_excel(excel_file, 'SteilingH44003_smoking')
food = pd.read_excel(excel_file, 'food')
housing = pd.read_excel(excel_file, 'housing')
thrive_survey = pd.read_excel(excel_file, 'thrive')
active_meds = pd.read_excel(excel_file, 'SteilingH44003_activemeds')
ct_appointments = pd.read_excel(excel_file, 'SteilingH44003_CT')
ct_appointments_2 = pd.read_excel(excel_file2, 'Sheet1')
pathology = pd.read_excel(excel_file, 'SteilingH44273_pathology')
follow_up_appointments = pd.read_excel(excel_file, 'SteilingH44003_followupappt')
procedures_dates = pd.read_excel(excel_file, 'SteilingH44273_procedures')
notes = pd.read_excel(excel_file, 'SteilingH44273_notes')
mental_illness_screens = pd.read_excel(excel_file, 'SteilingH44273_BHScreens')

In [6]:
def list_to_binary_series(l, index):
    s = pd.Series(False, index=index)
    s[l] = True
    return s

def series_of_delimited_strings_to_binary_table(series, **kwargs):
    delimiter = kwargs.get('delimiter', ' ')
    series_of_lists = series \
        .map(lambda x: x.split(delimiter)) \
        .map(lambda x: [i.strip() for i in filter(lambda y: y != '', x)]) # Make CONSULT_ORDERED data type a list instead of a delimited string
    index = set()
    for l in series_of_lists:
        index.update(l)
    return pd.DataFrame({'lists': series_of_lists}) \
        .apply(lambda x: list_to_binary_series(x['lists'], sorted(index)), axis=1)

In [7]:
# transform data in a column of a dataframe, using a dictionary as instructions
def cleanup_column(df, colname, dictionary):
    def transform(data, dictionary):
        if data in dictionary:
            return dictionary[data]
        return dictionary['Default'](data)
    df[colname] = df[colname].map(lambda x: transform(x, dictionary))

# search DX_NAME for keywords that might indicate a problem (flag)
# Convert DX_NAME from string data to two binary columns
def dx_name_flagger(string):
    housing_flag = not (re.search('hous|adobe', string, re.I) is None)
    homeless_flag = not (re.search('homeless', string, re.I) is None)
    economic_flag = not (re.search('finan|econom', string, re.I) is None)
    return pd.Series([homeless_flag, housing_flag or economic_flag], index=['DX_NAME ' + s for s in ['HOMELESS', 'HOUSING_OR_ECONOMIC']])

In [8]:
# a dictionary whose keys are data fields (column name) of patient info and values are transform maps
# to be used with the cleanup_column utility function
# the point is to consolidate categories that mean the same thing
# the Default field is the catch-all data handler
cleanup_dictionary_patient_info = {
    'hispanicindicator': {
        'Hispanic, Latino, Latina, Latinx, or of Spanish or Latin American origin ': 'Yes',
        'Not Hispanic, Latino, Latina, Latinx, or of Spanish or Latin American origin ': 'No',
        'Default': lambda x: 'Unknown'
    },
    'sexassignedatbirth': {
        'Female': 'Female',
        'Male': 'Male',
        'Default': lambda x: 'Unknown'
    },
    'maritalstatus': {
        '*Unspecified': 'Unknown',
        'Default': lambda x: x
    },
    'preferredlanguage': {
        'English': 'English',
        '*Unspecified': 'Unknown',
        'Default': lambda x: 'Other'
    },
    'HighestLevelOfEducation': {
        'Declined': 'Unknown',
        '*Unspecified': 'Unknown',
        'Unavailable': 'Unknown',
        'Default': lambda x: x
    },
    'HOUSING_SCREEN': {
        ('I do not have a steady place to live '
         '(I am temporarily staying with others, in a hotel, '
         'in a shelter, living outside on the street, on a bench, '
         'in a car, abandoned building, bus or train station, or in a park)'): 'No',
        ('I have a place to live today, '
         'but I am worried about losing it in the future'): 'Insecure',
        'Default': lambda x: x
    },
}

# combining all patient unique attributes/flags from multible sheets to a single dataframe
patient_info = demographic.rename(columns={'id': 'ID'}) \
    .merge(thrive_survey, on='ID', how='left') \
    .merge(food.drop_duplicates(['ID'], keep='last'), on='ID', how='left') \
    .merge(housing, on='ID', how='left') \
    .fillna({'cdwrrace': 'Unknown', 
             'HOUSING_SCREEN': 'Unknown', 
             'CONSULT_ORDERED': '',
             'FOOD_PANTRY_VST': 0, # WARNING: assuming NaN values are zero
             'SDOH_FOOD': 0, # WARNING: assuming NaN values are zero
             'SDOH_FOODBANK': 0, # WARNING: assuming NaN values are zero
             'FP_REF': 0, # WARNING: assuming NaN values are zero
             'DX_NAME': ''
            })

# drop singleton/useless data fields
patient_info = patient_info.drop(columns=[col for col in patient_info if len(patient_info[col].unique()) == 1])

# consolidate various data fields using instructions from the cleanup_dictionary
for colname, dictionary in cleanup_dictionary_patient_info.items():
    cleanup_column(patient_info, colname, dictionary)
    
# transform binary columns with data in the form of [null <something>] to [False True]
binary_columns = [col for col in patient_info if len(patient_info[col].unique()) == 2]
patient_info[binary_columns] = patient_info[binary_columns].isna().map(lambda x: not x)
# these two columns has NaN as positive indicator, renamed for clarity
patient_info = patient_info.rename(columns={'REG_ADDRESS': 'NOT_REG_ADDRESS', 'HX_ADDRESS': 'NOT_HX_ADDRESS'}) 

# Splitting CONSULT_ORDERED into multiple binary columns
consult_ordered_table = series_of_delimited_strings_to_binary_table(patient_info['CONSULT_ORDERED'], delimiter=';')
consult_ordered_table.insert(loc=0, column='ID', value=patient_info['ID'])
patient_info = patient_info.drop(columns='CONSULT_ORDERED') \
    .merge(consult_ordered_table, on='ID', how='left') \
    .rename(columns={x: 'CONSULT_ORDERED' + x.split(':')[1] for x in consult_ordered_table.columns if x != 'ID'})

# Consolidate and transform DX_NAME to multiple binary columns
dx_name_table = patient_info.apply(lambda x: dx_name_flagger(x['DX_NAME']), axis=1)
dx_name_table.insert(loc=0, column='ID', value=patient_info['ID'])
patient_info = patient_info.drop(columns='DX_NAME') \
    .merge(dx_name_table, on='ID', how='left')

# Perform one hot encoding to categorical data
patient_info = pd.get_dummies(patient_info)

for col in patient_info:
    print('{}: {}'.format(col, patient_info[col].unique()))

ID: [   1    2    3 ... 6101 6102 6103]
ageinyears: [52 59 68 71 69 64 74 79 57 72 77 73 53 67 76 63 65 81 62 70 75 66 80 61
 82 60 56 58 51 55 78 85 84 83 54]
FOOD_POS: [False  True]
FOOD_EMERG: [False  True]
TROUBLE_MEDS: [False  True]
TROUBLE_TRANS: [False  True]
TROUBLE_UTILS: [False  True]
TROUBLE_CARE: [False  True]
UNEMPLOY: [ True False]
MORE_EDU: [False  True]
FOOD_RES: [False  True]
CHILD_RES: [False  True]
SENIOR_RES: [False  True]
DISABLED_RES: [False  True]
UTIL_RES: [False  True]
TRANS_RES: [False  True]
EDU_RES: [False  True]
EMP_RES: [False  True]
MED_RES: [False  True]
HOUSING_RES: [False  True]
ICD_10: [False  True]
HVS_POS: [False  True]
EMERG: [False  True]
FOOD_PANTRY_VST: [  1.   0.   3.   9.   2.   4.  38.  12.  35.  21.  11.  52.   5.  25.
  24.  54.  59.   7.  13.  10.  69.  45.  18.  76.  33.  15.  79.  16.
 112.   6.  26.  29.  49.  40.  32.  72.  17.  61.  67.   8.  46.  14.
  22.  19.  30.  31.  64.  23. 172.  37.  42.  56.  20.  51. 116.  95.
  50.  34.  3

In [9]:
# consolidate data from comorb_problist and comorb_encounter
# drops codeDescription and simplify Code from <CodeType><CodeNumber>.<MinorCode> to just <CodeType>
comorb_list = pd.concat([comorb_problist, comorb_encounter]) \
    .drop_duplicates() \
    .groupby(by='id') \
    .apply(lambda x: pd.Series([' '.join(set(x['Code'].map(lambda x: x[0])))], index=['Code']), include_groups=False) \
    .reset_index()

# generate missing patient id's as empty code list
comorb_list = comorb_list.merge(pd.DataFrame({'id': patient_info['ID']}), on='id', how='right') \
    .fillna({'Code': ''})

# Create descriptions for simplified Code by combining all non-duplicate keywords across all descriptions that share the same CodeType
# This is for reference purposes only
def consolidate_CodeDescription(desc):
    result = []
    for d in desc:
        d = re.sub('[()]', '', d).lower()
        result += [token.lemma_ for token in nlp(d) if not token.is_stop and not token.is_punct]
    n_token = len(result)
    result = [(token, count / n_token) for i, (token, count) in enumerate(Counter(result).most_common()) if i < 5]
    return pd.Series([result], index=['Desc'])

# just a table for human use
simplified_code_descriptions = pd.concat([comorb_problist, comorb_encounter])[['Code', 'CodeDescription']] \
    .drop_duplicates() \
    .apply(lambda x: pd.Series([x['Code'][0], x['CodeDescription']], index=x.index), axis=1) \
    .groupby('Code') \
    .apply(lambda x: consolidate_CodeDescription(x['CodeDescription']), include_groups=False)

# create comorb_table from comorb_list, basically turn categorical data into binary columns
comorb_table = series_of_delimited_strings_to_binary_table(comorb_list['Code'], delimiter=' ')
comorb_table.insert(loc=0, column='ID', value=comorb_list['id'])
comorb_table

,ID,C,D,F,G,I,J,Z
0,1,False,False,True,False,False,False,False
1,2,False,False,True,False,False,False,False
2,3,False,True,True,False,False,False,True
3,4,False,True,True,False,False,True,False
4,5,False,True,True,False,False,False,True
...,...,...,...,...,...,...,...,...
6098,6099,False,False,False,False,False,False,False
6099,6100,False,False,False,False,False,False,False
6100,6101,False,False,False,False,False,False,False
6101,6102,False,False,True,False,False,False,False


In [10]:
# only look at rows that are total PHQ9 Scores
# then sort the scores into severity and treat as categorical data
# also keep RecordedDate to match with appointments
def process_PHQ9_tables(df):
    df = df.sort_values('RecordedDate').reset_index(drop=True)[['RecordedDate', 'Value']]
    df['Value'] = df['Value'].astype(int).map(lambda x: 'Severe' if x >= 15 else 'Moderate' if x >= 10 else 'Negligible')

    # Define all possible categories that can exist
    all_categories = ['Negligible', 'Moderate', 'Severe']
    # Convert the 'Value' column to a Categorical type
    df['Value'] = pd.Categorical(df['Value'], categories=all_categories)
    # One hot encoding for 'Value'
    df = pd.get_dummies(df, columns=['Value'], drop_first=True)
    return df

PHQ9_scores = (
    mental_illness_screens[
        mental_illness_screens['DisplayName'].map(lambda x: not (re.search('PHQ-9 Score', x) is None))
        & ~mental_illness_screens['Value'].isna()
    ]
    .rename(columns={'id': 'ID'})
    .reset_index(drop=True)
    .groupby('ID')
    .apply(process_PHQ9_tables, include_groups=False)
)

PHQ9_scores

RecordedDate  Value_Moderate  Value_Severe
ID                                                      
2    0 2019-08-13 17:43:00           False          True
5    0 2022-11-15 15:59:00           False         False
23   0 2022-03-16 09:00:00           False         False
     1 2023-04-05 13:15:00           False         False
29   0 2022-12-13 20:12:00            True         False
...                    ...             ...           ...
6030 0 2022-06-01 14:13:00           False         False
6045 0 2023-07-22 10:41:00            True         False
     1 2023-08-17 16:14:00           False         False
6065 0 2023-06-21 19:25:00           False         False
6102 0 2023-10-23 15:45:00           False         False

[2336 rows x 3 columns]

In [11]:
# Generate cleanup dictionary for 'ordering deptname'
general_internal_medicine = {x: 'General Internal Medicine' for x in """
CRO PRIMARY CARE 5A
CRO PRIMARY CARE 6B
CRO FAMILY MEDICINE
CRO PRIMARY CARE 5B
CRO PRIMARY CARE 6A
CRO PRIMARY CARE 6C
CRO PRIMARY CARE 5C
CRO PRIMARY CARE
SHA PRIMARY CARE 5A
SHA PRIMARY CARE 5B
SHA PRIMARY CARE 6B
SHA PRIMARY CARE 6C
SHA PRIMARY CARE 5C
SHA PRIMARY CARE 6A
SHA OBAT
CRO OBAT MEN
CRO ILI 2
""".split('\n') if x != ''}

family_medicine = {x: 'Family Medicine' for x in """
CRO FAMILY MEDICINE OBAT
CRO FAM MED START
YAW FAMILY MEDICINE
""".split('\n') if x != ''}

geriatrics = {x: 'Geriatrics' for x in """
SHA GERIATRIC PRACTICE
BMC OFFSITE GERIATRICS
BMC OFFSITE GERIATRICS NURSING HOME
ONH MARIAN MANOR FAMILY MEDICINE(NH)
""".split('\n') if x != ''}

infectious_disease = {x: 'Infectious Disease' for x in """
SHA CENTER FOR INF DISEASE
SHA ID CONSULT
""".split('\n') if x != ''}

ob_gyn_outpatient = {x: 'Ob/Gyn Outpatient' for x in """
YAW MAT FASTER PATHS
YAW OB/GYN MAIN
YAW GYN SPECIALTY
YAW BEACON
""".split('\n') if x != ''}

comorb_lung = {'SHA RHEUMATOLOGY CLINIC': 'Patient with other lung problems'}

comorb_cancer = {x: 'Patient with other cancers' for x in """
SHA OMFS
YAW OMFS CLINIC
SHA UROLOGY
MOA HEM ONC
MOA ONCOLOGY
MOA OTOLARYNGOLOGY
MOA RAD ONC
MOA BELKIN BREAST HEALTH
MOA MULTI SPECIALT SURGERY
MOA HEMATOLOGY
BMC ANESTHESIA
BMC HEMATOLOGY
""".split('\n') if x != ''}

pulmonary_allergy_critcare = {x: 'Pulmonary, Allergy, and Critical Care' for x in """
SHA PULMONARY CLINIC
SHA ALLERGY CLINIC
MOA PULM THORACIC
BMC TOBACCO TREATMENT CONSULT SERVICE
PRE CARD THOR SURG CLN
MOA THORACIC ONCOLOGY
""".split('\n') if x != ''}

community_health_center = {x: 'Community Health Center' for x in """
OCH EAST BOSTON HC
OCH ANC/SOUTOCH BOSTON CHC
OCH UPHAMS CORNER HC
OCH MCINNIS HEALTH GROUP
OCH WHITTIER ST NHC
OCH CODMAN SQUARE HC
OCH DORCHESTER HOUSE HC
OCH DH RAD
DHH ADULT MED
DHH FAMILY MED
DHH ID
OCH MATTAPAN COMM HC
OCH SOUTH BOS RAD
CSHC INTERNAL MED
CSHC FAMILY MED
CSHC PEDIATRICS
OHC HARBOR CHC
OHC LOWELL HC
OHC LYNN CHC
OHC SOUTH COVE HANCOCK
OHC DIMOCK
OHC MANET HC OCM COPLEY AT CHARLES RIVER
OHC HARVARD ST
OCM BU CHARLES RIVER
OCM COMMUNITY MED GROUP
BUAP COPLEY MP
""".split('\n') if x != ''}

emergency_care = {x: 'Emergency care, urgent care, or inpatient services' for x in """
MEN EMERGENCY DEPT
MEN 6W MEDICAL
MEN 7 EAST
MEN 7E SURGICAL
ENP 6W MEDICAL UNIT
MEN ICU
MICU FAMILY MED POSTING
BMC HOSPITALISTS
MEN LABOR AND DELIVERY
YAW LABOR AND DELIVERY
MEN NEWBORN NURSERY
YAW NURSERY
YAW 5 PEDI
DHH URGENT CARE
""".split('\n') if x != ''}

radiology_services = {x: 'Radiology Services' for x in """
MEN RAD CT
MEN RAD MRI
MEN RAD US
MEN RAD NUCLEAR MEDICINE
ZZZMEN RADIOLOGY *Unspecified
""".split('\n') if x != ''}

# other_deps = {x: 'Other' for x in """
# CRO ADULT PSYCHIATRY
# DOW ADULT PSYCHIATRY
# SHA GASTROENTRLGY BUMG
# SHA GASTRO LIVER
# SHA RENAL MEDICINE CLINIC
# SHA TRANSPLANT SURGERY
# SHA NEUROSURGERY
# SHA NEUROLOGY
# PRE ENDOCRINOLOGY
# PRE ENDO/NUTR
# PRE OCCUPATIONAL THERAPY
# MOA AMYLOID
# PRE CARDIOLOGY
# YAW PEDIATRIC CLINIC
# BMC OFFSITE PEDI
# BMC EPICCARE LINK
# DHH MANAGED CARE
# DHH SPECIAL CLINIC
# STABILIZATION CARE CTR
# TRANSITIONAL CARE CTR
# """.split('\n') if x != ''}

cleanup_deps = general_internal_medicine | family_medicine | geriatrics \
    | infectious_disease | ob_gyn_outpatient | comorb_lung | comorb_cancer \
    | pulmonary_allergy_critcare | community_health_center | emergency_care | radiology_services \
    | {'Default': lambda x: 'Other'}

In [12]:
ignore_columns_screening_appointments = [
    'orderprocedurename',
    'OrderedDateKey', 'CDWRPrimaryCategoryInsurance', 
    'CDWRSecondaryCategoryInsurance', 'packyears', 'yearssincequit',
    'asymptomatic', 'initial_followup', 'AMBhistcancer', 'attest',
    'RIShistcancer', 'VisitType (Visit redulting from order)',
    'CTExamStartInstant', 'accessionnumber', 'studystatus', 
    'impression', 'birads'
]

cleanup_dictionary_screening_appointments = {
    'CDWRPrimarySubcategoryInsurance': {
        'Medicare': 'Medicare',
        'Medicaid': 'Medicaid',
        'Self-Pay': 'Self-Pay',
        'Default': lambda x: 'Other'
    },
    'CDWRSecondarySubcategoryInsurance': {
        'Medicare': 'Medicare',
        'Medicaid': 'Medicaid',
        'Self-Pay': 'Self-Pay',
        'Default': lambda x: 'Other'
    },
    'ordering deptname': cleanup_deps
}

date_of_dataset = pd.to_datetime('2025-07-15')
source = ct_appointments
screening_appointments = source[
        # We only care about screening appointments
        source['orderprocedurename'].map(lambda x: x == 'CT LUNG SCREENING')
        # We only care about 'Completed', 'No Show', Canceled' appointments
        & source['AppointmentStatus'].map(lambda x: x in ['Completed', 'No Show', 'Canceled'])
        # filter out appointmnets with no schedule time
        & ~source['VisitScheduledDateTime (Visit redulting from order)'].dt.time.isna()
        # filter out appointments with schedule time occuring after appointment time (which makes no sense)
        & (source['VisitScheduledDateTime (Visit redulting from order)'] < ct_appointments['AppointmentDateTime (Visit redulting from order)'])
        # filter out appointments with appointment time after the day of dataset collection since we don't know the future
        & (source['AppointmentDateTime (Visit redulting from order)'] < date_of_dataset)
    ] \
    .drop(columns=ignore_columns_screening_appointments) \
    .fillna({'packyears': 0, 'current_former_smoker': 'Unknown/Never'}) \
    .rename(columns={'id': 'ID'})

for colname, dictionary in cleanup_dictionary_screening_appointments.items():
    cleanup_column(screening_appointments, colname, dictionary)
    
#####################
# The following steps extract features from 'VisitScheduledDateTime (Visit redulting from order)', and 'AppointmentDateTime (Visit redulting from order)'
# How many days between the scheduling of an appointment and the day of the appointment itself
screening_appointments['days_scheduled_ahead'] = (
    screening_appointments['AppointmentDateTime (Visit redulting from order)']
    - screening_appointments['VisitScheduledDateTime (Visit redulting from order)']).dt.days

# Time of the day of an appointment
screening_appointments['time_of_day'] = screening_appointments['AppointmentDateTime (Visit redulting from order)'].dt.time \
    .map(lambda x: 'Before 8AM' if x < datetime.time(8)
        else '8AM-10AM' if x < datetime.time(10)
        else '10AM-12PM' if x < datetime.time(12)
        else '12PM-2PM' if x < datetime.time(14)
        else '2PM-4PM' if x < datetime.time(16)
        else 'After 4PM')

# Day of the week of an appointment
screening_appointments['appointment_day_of_week'] = screening_appointments['AppointmentDateTime (Visit redulting from order)'].dt.strftime('%A')
# Month of an appointment
screening_appointments['appointment_month'] = screening_appointments['AppointmentDateTime (Visit redulting from order)'].dt.strftime('%B')

# whether the previous appointment is canceled or no show
def flag_previous_appointment(df):
    df = df.sort_values('AppointmentDateTime (Visit redulting from order)').reset_index(drop=True)
    prev_app_failed = [False]
    prev_app_failed[1:] = [x != 'Completed' for x in df['AppointmentStatus'][:-1]]
    df['FailedPreviousApp'] = pd.Series(prev_app_failed)
    return df
screening_appointments = screening_appointments.groupby('ID') \
    .apply(flag_previous_appointment, include_groups=False) \
    .reset_index().drop(columns='level_1')

# Retrieve PHQ9 score from the closest test taken before the appointment date
def retrieve_PHQ9(data, PHQ9_data):
    ID = data['ID']
    app_datetime = data['AppointmentDateTime (Visit redulting from order)']
    # treat patients with no PHQ9 screening as if they scored 0 on a screening a very long time ago
    if ID not in PHQ9_data.index.get_level_values(0):
        return pd.Series([36524, False, False], index=['DaysFromPHQ9', 'PHQ9_Moderate', 'PHQ9_Severe'])
                         
    screenings = PHQ9_data.loc[data['ID']]
    recorded_date = screenings['RecordedDate']
    delta_time = (app_datetime - recorded_date).abs()
    closest_screening_idx = delta_time.idxmin()
    closest_screening = screenings.loc[closest_screening_idx]
    return pd.Series([
        delta_time[closest_screening_idx].days, 
        closest_screening['Value_Moderate'], 
        closest_screening['Value_Severe']
    ], index=['DaysFromPHQ9', 'PHQ9_Moderate', 'PHQ9_Severe'])

PHQ9_table = screening_appointments.apply(lambda x: retrieve_PHQ9(x, PHQ9_scores), axis=1)
screening_appointments = screening_appointments.merge(PHQ9_table, left_index=True, right_index=True, how='left')

# fetching weather data
latitude = 42.3601
longitude = -71.0589
start_date = screening_appointments['AppointmentDateTime (Visit redulting from order)'].min().strftime("%Y-%m-%d")
end_date = screening_appointments['AppointmentDateTime (Visit redulting from order)'].max().strftime("%Y-%m-%d")

# Construct the API URL for the date range
url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={latitude}&longitude={longitude}&"
    f"start_date={start_date}&end_date={end_date}&"
    f"daily=temperature_2m_mean,precipitation_sum" # Requesting daily summaries
)

# Make a single API request for the entire date range
response = requests.get(url)
data = response.json()

# Process the weather data with pandas
weather_df = pd.DataFrame(data=data['daily'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df.set_index('time', inplace=True)
appointment_weather = weather_df.loc[screening_appointments['AppointmentDateTime (Visit redulting from order)'].dt.date].reset_index(drop=True)
screening_appointments = pd.concat([screening_appointments, appointment_weather], axis=1)

# All relevant information is extracted, remove these columns
screening_appointments_old = screening_appointments
screening_appointments = screening_appointments.drop(columns=['VisitScheduledDateTime (Visit redulting from order)', 'AppointmentDateTime (Visit redulting from order)'])

# One hot encoding of all categorical data
appointment_outcome = screening_appointments['AppointmentStatus']
screening_appointments = pd.get_dummies(screening_appointments.drop(columns='AppointmentStatus'))
# for col in screening_appointments:
#     print('{}: {}'.format(col, screening_appointments[col].unique()))

In [13]:
screening_appointments.loc[0:10]

,ID,days_scheduled_ahead,FailedPreviousApp,DaysFromPHQ9,PHQ9_Moderate,PHQ9_Severe,temperature_2m_mean,precipitation_sum,ordering deptname_Community Health Center,"ordering deptname_Emergency care, urgent care, or inpatient services",...,appointment_month_December,appointment_month_February,appointment_month_January,appointment_month_July,appointment_month_June,appointment_month_March,appointment_month_May,appointment_month_November,appointment_month_October,appointment_month_September
0,1,15,False,36524,False,False,0.2,0.2,False,False,...,True,False,False,False,False,False,False,False,False,False
1,1,15,False,36524,False,False,0.2,0.2,False,False,...,True,False,False,False,False,False,False,False,False,False
2,1,2,False,36524,False,False,-4.1,0.0,False,False,...,False,True,False,False,False,False,False,False,False,False
3,1,2,False,36524,False,False,-4.1,0.0,False,False,...,False,True,False,False,False,False,False,False,False,False
4,1,2,False,36524,False,False,-4.1,0.0,False,False,...,False,True,False,False,False,False,False,False,False,False
5,2,55,False,1280,False,True,4.0,0.1,False,False,...,False,True,False,False,False,False,False,False,False,False
6,2,55,True,1280,False,True,4.0,0.1,False,False,...,False,True,False,False,False,False,False,False,False,False
7,2,8,True,1312,False,True,7.7,1.0,False,False,...,False,False,False,False,False,True,False,False,False,False
8,2,8,True,1312,False,True,7.7,1.0,False,False,...,False,False,False,False,False,True,False,False,False,False
9,2,270,True,1574,False,True,7.3,6.4,False,False,...,True,False,False,False,False,False,False,False,False,False


In [14]:
def clean_column_names(df):
    """
    Cleans column names to be readable and LightGBM compatible.
    """
    original_columns = df.columns.tolist()
    cleaned_columns = []
    
    for col in original_columns:
        # 1. Add a space before capital letters (for camelCase)
        col = re.sub(r'([a-z])([A-Z])', r'\1 \2', col)
        # 2. Replace slashes, commas, and underscores with a space
        col = re.sub(r'[/,_]+', ' ', col)
        # 3. Shorten common long phrases
        col = col.replace('ordering deptname', 'dept')
        col = col.replace('HighestLevelOfEducation', 'education')
        # 4. Collapse multiple spaces into one and strip whitespace
        col = re.sub(r'\s+', ' ', col).strip()
        # 5. Replace spaces with underscores and convert to lowercase (snake_case)
        col = col.replace(' ', '_').lower()
        
        cleaned_columns.append(col)
        
    df.columns = cleaned_columns
    return df

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

def run_evaluation_suite(model, X_test, y_test, positive_class_label, category_mapping=None):
    """
    Generates an evaluation suite for a binary classification model.
    Handles the integer output of XGBoost models by decoding them.

    Args:
        model: The trained classifier.
        X_test (DataFrame or array): The test features.
        y_test (Series or array): The true test labels (as strings/categories).
        positive_class_label (str): The name of the positive class for ROC analysis.
        category_mapping (dict, optional): Required only if the model is XGBoost.
                                            Maps integer predictions to category names.
    """
    y_pred = None

    # --- 1. Get Predictions (with special handling for XGBoost) ---
    if isinstance(model, xgb.XGBClassifier):
        if category_mapping is None:
            raise ValueError("A 'category_mapping' must be provided for XGBoost models.")
        # Predict integers and map them back to categories
        int_preds = model.predict(X_test)
        y_pred = pd.Series(int_preds).map(category_mapping)
    else:
        # For other models like RF, predict directly
        y_pred = model.predict(X_test)

    # --- 2. Classification Report ---
    print("--- Classification Report ---")
    print(classification_report(y_test, y_pred))

    # --- 3. Confusion Matrix ---
    print("\n--- Confusion Matrix ---")
    # Ensure consistent order of labels in the matrix
    class_labels = np.unique(y_test)
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_labels,
                yticklabels=class_labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

    # --- 4. ROC Curve and AUC Score ---
    # Get the probabilities for the positive class
    pos_label_index = -1
    
    # FIX: Check if model.classes_ are integers (like XGBoost) or strings
    if np.issubdtype(model.classes_.dtype, np.integer):
        # Invert the mapping to find the integer code for the positive class string
        inverse_mapping = {v: k for k, v in category_mapping.items()}
        pos_label_int = inverse_mapping[positive_class_label]
        pos_label_index = np.where(model.classes_ == pos_label_int)[0][0]
    else:
        # For other models, classes_ are strings
        pos_label_index = np.where(model.classes_ == positive_class_label)[0][0]
        
    y_pred_proba = model.predict_proba(X_test)[:, pos_label_index]
    
    # Calculate ROC curve metrics
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba, pos_label=positive_class_label)
    roc_auc = auc(fpr, tpr)
    print(f"\nAUC Score for '{positive_class_label}': {roc_auc:.4f}")

    # Plot the ROC Curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for "{positive_class_label}" Class')
    plt.legend(loc="lower right")
    plt.grid()
    plt.show()

In [16]:
seed = 200
X = clean_column_names(screening_appointments.merge(patient_info, on='ID', how='left') \
    .merge(comorb_table, on='ID', how='left') \
    .drop(columns='ID'))
# y = appointment_outcome.astype('category')
y = appointment_outcome.map(lambda x: 'No Show' if x == 'No Show' else 'Complete/Canceled').astype('category')
category_mapping = dict(enumerate(y.cat.categories))

In [17]:
def get_smote_rfe_ranking(
    classifier,
    importance_attribute: str,
    X,
    y,
    cv=5,
    scoring='roc_auc',
    min_features_to_select=1,
    step=1,
    n_jobs=-1,
    random_state=42,
    # New Caching Parameters
    cache_filepath: str = None,
    use_cache: bool = False,
):
    """
    ... (Docstring truncated)
    """

    # --- Caching Load Check ---
    if use_cache and cache_filepath and os.path.exists(cache_filepath):
        print(f"Loading cached results from {cache_filepath}...")
        try:
            # Load the tuple (feature_ranking, rfecv_selector)
            feature_ranking, rfecv_selector = joblib.load(cache_filepath)
            print("Successfully loaded cached results.")
            return feature_ranking, rfecv_selector
        except Exception as e:
            print(f"Error loading cache: {e}. Re-running RFECV.")
            # If load fails, fall through to re-run the process

    # --- 1. Set up the RFE Estimator Pipeline ---
    # ... (Your existing setup code for clf, estimator_pipeline, full_importance_getter) ...
    clf = clone(classifier)
    try:
        clf.set_params(random_state=random_state)
    except ValueError:
        print(f"Note: {classifier.__class__.__name__} "
              "does not have a 'random_state' parameter. Proceeding without it.")
        pass

    estimator_pipeline = ImPipeline([
        ('smote', SMOTE(random_state=random_state)),
        ('classifier', clf)
    ])
    
    full_importance_getter = f'named_steps.classifier.{importance_attribute}'

    rfecv_selector = RFECV(
        estimator=estimator_pipeline,
        min_features_to_select=min_features_to_select,
        step=step,
        cv=cv,
        scoring=scoring,
        importance_getter=full_importance_getter,
        n_jobs=n_jobs
    )

    # --- 2. Run the Feature Elimination to find ranks and mask ---
    print("Running Recursive Feature Elimination...")
    rfecv_selector.fit(X, y) # This is the long step
    print("Done.")

    # --- 3. Perform Explicit Final Fit on Selected Features ---
    # ... (Your existing code for setting up final_model_pipeline and fitting X_selected) ...
    selected_features_mask = rfecv_selector.support_
    selected_feature_names = X.columns[selected_features_mask]
    X_selected = X[selected_feature_names]

    final_clf = clone(classifier)
    try:
        final_clf.set_params(random_state=random_state)
    except ValueError:
        pass
        
    final_model_pipeline = ImPipeline([
        ('smote', SMOTE(random_state=random_state)),
        ('classifier', final_clf)
    ])

    print("Fitting final model on selected features...")
    final_model_pipeline.fit(X_selected, y)
    print("Done.")

    # --- 4. Get Importances from the Explicit Final Model & 5. Create the Combined Ranking ---
    # ... (Your existing code for calculating final_importances and feature_ranking DataFrame) ...
    classifier_step = final_model_pipeline.named_steps.classifier

    try:
        importances = getattr(classifier_step, importance_attribute)
        
        if importances.ndim > 1:
            final_importances = np.abs(importances).sum(axis=0)
        else:
            final_importances = np.abs(importances).ravel()
            
    except AttributeError:
        raise AttributeError(
            f"The classifier {classifier.__class__.__name__} "
            f"does not have the attribute '{importance_attribute}'."
        )
        
    full_ranking = pd.DataFrame({
        'feature': X.columns,
        'rfe_rank': rfecv_selector.ranking_
    })

    selected_ranking = pd.DataFrame({
        'feature': selected_feature_names,
        'final_importance': final_importances
    })

    df_selected = full_ranking[full_ranking['rfe_rank'] == 1].merge(selected_ranking, on='feature')
    df_eliminated = full_ranking[full_ranking['rfe_rank'] > 1]

    df_selected = df_selected.sort_values(by='final_importance', ascending=False)
    df_eliminated = df_eliminated.sort_values(by='rfe_rank', ascending=True)

    feature_ranking = pd.concat([df_selected, df_eliminated])
    feature_ranking = feature_ranking.reset_index().rename(columns={'index': 'original_index'})
    feature_ranking = feature_ranking[['original_index', 'feature', 'rfe_rank', 'final_importance']]
    
    # --- Caching Save Step ---
    if cache_filepath:
        print(f"Saving results to cache file {cache_filepath}...")
        
        # Get the directory path from the file path
        cache_dir = os.path.dirname(cache_filepath)
        
        # Check if the directory is not empty (i.e., not saving to the current directory)
        if cache_dir: 
            try:
                # Create the directory recursively if it doesn't exist
                os.makedirs(cache_dir, exist_ok=True)
                print(f"Ensured cache directory exists at: {cache_dir}")
            except OSError as e:
                # Handle potential permission or path errors gracefully
                print(f"Error creating cache directory {cache_dir}: {e}. Proceeding with dump, which may fail.")

        # Save the returned objects as a tuple
        joblib.dump((feature_ranking, rfecv_selector), cache_filepath)
        print("Save complete.")

    return feature_ranking, rfecv_selector

In [18]:
feature_ranking, rfecv_selector = get_smote_rfe_ranking(
    RandomForestClassifier(random_state=seed+1, class_weight='balanced'),
    'feature_importances_',  # e.g., 'feature_importances_' or 'coef_'
    X,
    y, 
    cache_filepath="cache/rfecv_selector", 
    use_cache=True
)

Loading cached results from cache/rfecv_selector...
Successfully loaded cached results.


In [19]:
print_and_plot_rfe_results(feature_ranking, rfecv_selector)

NameError: name 'print_and_plot_rfe_results' is not defined

In [ ]:
features = feature_ranking.loc[0:44]['feature']
X_train, X_test, y_train, y_test = train_test_split(X[features], y, test_size=0.3, random_state=seed)
print('Original dataset shape %s' % Counter(y_train))


# Apply SMOTE to training data
smote = SMOTE(random_state=seed)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize the undersampler
# rus = RandomUnderSampler(random_state=seed)
# X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

# X_train_resampled, y_train_resampled = X_train, y_train

print('Resampled dataset shape %s' % Counter(y_train_resampled))

# X_train_resampled, y_train_resampled = (X_train, y_train)
print(X_train.shape)

In [ ]:
# Initialize the Random Forest model
# n_estimators is the number of trees; random_state ensures reproducibility
rf_model = RandomForestClassifier(n_estimators=200, random_state=seed, class_weight='balanced')

# Train the model on the training data
rf_model.fit(X_train_resampled, y_train_resampled)

In [ ]:
# Make predictions on the test set
y_pred = rf_model.predict(X_test)

# Evaluate the model's accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
# Expected Output: Model Accuracy: 1.0000

In [ ]:
run_evaluation_suite(rf_model, X_test, y_test, 'No Show', category_mapping)

In [ ]:
# Check feature importances
rf_importances = rf_model.feature_importances_
rf_feature_importances = pd.Series(rf_importances, index=X_train.columns)
rf_feature_importances.nlargest(10).plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# 2. Initialize and train the model
# For a regression task, use lgb.LGBMRegressor()
lgbm = lgb.LGBMClassifier(random_state=seed)
lgbm.fit(X_train_resampled, y_train_resampled)

In [ ]:
run_evaluation_suite(lgbm, X_test, y_test, 'No Show', category_mapping)

In [ ]:
# Extract and plot feature importances
feature_importances = pd.Series(lgbm.feature_importances_, index=X_train.columns)
feature_importances.nlargest(10).plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# --- 2. Initialize and train the XGBoost model ---
# For a regression task, use xgb.XGBRegressor()
xgb_model = xgb.XGBClassifier(random_state=seed)
xgb_model.fit(X_train_resampled, y_train_resampled.cat.codes)

In [ ]:
run_evaluation_suite(xgb_model, X_test, y_test, 'No Show', category_mapping)

In [ ]:
# --- 2. Get and sort feature importances ---
importances = xgb_model.feature_importances_
feature_importances = pd.Series(importances, index=X_train.columns)
feature_importances.nlargest(10).plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
lr_feature_ranking, lr_rfecv_selector = get_smote_rfe_ranking(
    LinearSVC(penalty='l1', dual=False, max_iter=10000, random_state=seed),
    'coef_',  # e.g., 'feature_importances_' or 'coef_'
    X,
    y,
    cache_filepath="cache/lr_rfecv_selector",
    use_cache=True
)

In [ ]:
print_and_plot_rfe_results(lr_feature_ranking, lr_rfecv_selector)

In [ ]:
lr_features = lr_feature_ranking.loc[:30]['feature']

In [ ]:
feature_candidates = lr_features[:30]
feature_interaction_indices = pd.Series(itertools.combinations(feature_candidates, 2))
feature_interaction_names = feature_interaction_indices.map(lambda i: f"{i[0]} x {i[1]}")
feature_interaction_indices = feature_interaction_indices.set_axis(feature_interaction_names)

bool_combo_categories = ['00', '01', '10', '11']
bool_cat_type = CategoricalDtype(categories=bool_combo_categories, ordered=True)
def generate_interactions(entry):
    def conditional_multiply(a, b):
        """
        Multiplies two values.
        - If both are bool, returns a combination code.
        - Otherwise, returns a float/int (standard multiplication).
        """
        if isinstance(a, bool) and isinstance(b, bool):
            return f"{int(a)}{int(b)}"
        else:
            return a * b
    return feature_interaction_indices.map(lambda i: conditional_multiply(entry[i[0]], entry[i[1]]))

X_interactions = X[lr_features].apply(generate_interactions, axis=1)
for col in X_interactions.columns:
    if X_interactions[col].dtype == 'object':
        X_interactions[col] = X_interactions[col].astype(bool_cat_type)
X_aug = pd.get_dummies(X[lr_features].join(X_interactions, how='left'), drop_first=True)

In [ ]:
X_aug_train, X_aug_test, y_aug_train, y_aug_test = train_test_split(X_aug, y, test_size=0.3, random_state=seed)
print('Original dataset shape %s' % Counter(y_aug_train))


# Apply SMOTE to training data
smote = SMOTE(random_state=seed)
X_aug_train_resampled, y_aug_train_resampled = smote.fit_resample(X_aug_train.astype(int), y_aug_train)

# Initialize the undersampler
# rus = RandomUnderSampler(random_state=seed)
# X_aug_train_resampled, y_aug_train_resampled = rus.fit_resample(X_aug_train, y_aug_train)

# X_train_resampled, y_train_resampled = X_train, y_train

print('Resampled dataset shape %s' % Counter(y_aug_train_resampled))

# X_train_resampled, y_train_resampled = (X_train, y_train)
print(X_aug_train.shape)

In [ ]:
# Create an instance of the LogisticRegression model
scalar_columns =  [i for i in X_aug.dtypes[X_aug.dtypes != 'bool'].index]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), scalar_columns)
    ],
    remainder='passthrough'
)
lr_model_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(penalty='l1', solver='liblinear', max_iter=20000))
])

param_grid = {
    'classifier__C': np.logspace(-4, 4, 9) # C values from 10^-4 to 10^4
}

grid_search = GridSearchCV(
    estimator=lr_model_base,
    param_grid=param_grid,
    scoring='roc_auc',  # Choose a performance metric suitable for your task
    cv=5,               # Use 5-fold cross-validation
    verbose=1           # Print progress
)
# Train the model on your data
grid_search.fit(X_aug_train_resampled, y_aug_train_resampled)
lr_model = grid_search.best_estimator_

In [ ]:
best_params = grid_search.best_params_
best_c = best_params['classifier__C']

In [ ]:
run_evaluation_suite(lr_model, X_aug_test, y_aug_test, 'No Show', category_mapping)

In [ ]:
# --- ANALYSIS 1: COEFFICIENT ANALYSIS ---
print("--- Coefficient-based Feature Importance ---")
coefs = lr_model.named_steps["classifier"].coef_[0]
feature_names = X_aug_train.columns
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
coef_df['Abs_Coefficient'] = np.abs(coef_df['Coefficient'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)
print_full(coef_df[0:10])
print("\n")

In [ ]:
print("--- Permutation-based Feature Importance ---")
# Using the scaled test set to prevent data leakage from the training process
result = permutation_importance(lr_model, X_aug_test, y_aug_test, n_repeats=10, random_state=seed)
perm_importances = result.importances_mean
perm_std = result.importances_std

perm_df = pd.DataFrame({'Feature': feature_names, 'Permutation_Importance': perm_importances})
perm_df = perm_df.sort_values(by='Permutation_Importance', ascending=False)
print(perm_df[0:20])

In [ ]:
lr_features